# Ba phương án bổ sung: Random Forest, SVM RBF, Gaussian Naive Bayes
Mở trong bản clone repo. Cài requirements-modeling-lock.txt trước khi chạy.
Notebook dùng cùng src/modeling_team.py với CLI; không tạo lại phép chia dữ liệu.

Đợt bổ sung diễn ra sau khi đã xem test LR/HGB. Đây là kết quả khám phá trên cùng test,
không phải xác nhận độc lập. Không sửa lưới hoặc ngưỡng theo kết quả test.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'src/modeling_team.py').exists():
    root = root.parent
if not (root / 'src/modeling_team.py').exists():
    raise RuntimeError('Mở notebook trong gốc repo hoặc notebooks/.')
sys.path.insert(0, str(root))
from IPython.display import display, Image, Markdown
from src.modeling import load_data
from src.modeling_team import additional_specs, model_scores, run
from src.preprocessing import FEATURES
from src.evaluation import split_table


## 1. Kiểm tra dữ liệu và ba pipeline bổ sung

In [ ]:
parts, groups = load_data()
display(split_table(parts))
for name, (pipeline, grid) in additional_specs().items():
    print(name, grid)
    display(pipeline)


## 2. Huấn luyện và so sánh chung
Chạy lại LR/HGB với lưới giữ nguyên và ba mô hình mới; CV 5 fold trên train theo nhóm.
Validation chỉ chọn ngưỡng; test tính sau khi cấu hình và ngưỡng đã chốt.
RF và SVM có thể mất vài phút. Kết quả được ghi vào reports/tables/12_*.csv.


In [ ]:
result = run()
display(result['cv'])
print('Mô hình được chọn bằng CV:', result['selected'])
display(result['test'])
display(result['comparisons'])


## 3. Biểu đồ và phần thuyết minh

In [ ]:
display(Image(filename=str(root / 'reports/figures/12_pr_roc.png')))
display(Image(filename=str(root / 'reports/figures/12_confusion.png')))
for filename in ['Bao_cao_Random_Forest.md', 'Bao_cao_SVM_RBF.md', 'Bao_cao_Gaussian_Naive_Bayes.md']:
    display(Markdown((root / 'reports' / filename).read_text(encoding='utf-8')))


## 4. Ví dụ nạp lại SVM cho demo
Điểm SVM là margin, không hiển thị thành phần trăm xác suất.
Dùng ngưỡng đã lưu; không dùng .predict() với ngưỡng mặc định.


In [ ]:
import joblib
bundle = joblib.load(root / 'outputs/team/svm_rbf.joblib')
example = parts['validation'][FEATURES].head(5)
score = model_scores(bundle['model'], example)
print('Loại điểm:', bundle['score_kind'])
print('Điểm SVM:', score)
print('Ngưỡng:', bundle['threshold'])
print('Cảnh báo:', score >= bundle['threshold'])
